<a href="https://colab.research.google.com/github/saithrr/HACKATON/blob/main/datos_simulados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SEMANA 1:** IMPORTACIÓN DE LIBRERÍAS Y SIMULACIÓN DE DATOS ALINEADA A LA LOGICA DEL EQUIPO DE BACKEND

In [10]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
import os

- Fijamos la semilla de datos simulados para que sean reproducibles

In [11]:
np.random.seed(42)
random.seed(42)
NUM_REGISTROS = 1000

- Creamos una base de datos simulada con la historia de 1000 inmuebles en LATAM

In [20]:
# Tarifa de referencia estandarizada por kWh

TARIFA_KWH = 0.75

# Lista de Inmuebles y Monedas en Regiones de LATAM


tipos_inmueble = ["Casa",
                  "Apartamento",
                  "Oficina",
                  "Comercio"]


regiones = ["USD",
            "MXN",
            "COP",
            "ARS",
            "CLP",
            "PEN",
            "BRL"]

# Diccionarios extraídos directamente de la lógica del Backend

baseline_por_tipo = {"Casa": 350,
                     "Apartamento": 220,
                     "Oficina": 500,
                     "Comercio": 700}

factor_clima = {"USD": 1.0,
                "MXN": 1.15,
                "COP": 1.05,
                "ARS": 1.10,
                "CLP": 0.90,
                "PEN": 0.95,
                "BRL": 1.20}


# Base datos de los inmuebles

datos = {
    "consumidor": [f"Usuario_{i}" for i in range(1, NUM_REGISTROS + 1)],
    "tipo_inmueble": np.random.choice(tipos_inmueble, NUM_REGISTROS),
    "moneda_region": np.random.choice(regiones, NUM_REGISTROS),
    "mes": np.random.randint(1, 13, NUM_REGISTROS),
    "dia_semana": np.random.randint(0, 7, NUM_REGISTROS),
    "uso_horario_pico": np.random.choice([1, 0], NUM_REGISTROS, p=[0.6, 0.4]),
    "cantidad_equipos": np.random.randint(1, 40, NUM_REGISTROS),
    "horas_alto_consumo": np.random.randint(1, 14, NUM_REGISTROS)
}

df = pd.DataFrame(datos)

- Simulación realista de los datos y la lógica de clasificación de perfiles.
 Determina si el usuario es Eficiente, Moderado o Ineficiente
 evaluando su comportamiento frente a lo que debería consumir de manera ideal.

In [22]:
def generar_consumo(fila):
    base = baseline_por_tipo[fila["tipo_inmueble"]]

    # Factor estacional: meses de verano o invierno extremo (1, 2, 7, 12) incrementan el uso

    factor_estacional = 1.15 if fila["mes"] in [1, 2, 7, 12] else 1.0

    # Ruido aleatorio realista (+/- 40%)

    variacion = np.random.uniform(0.6, 1.6)
    return round(base * factor_estacional * variacion, 2)

df["consumo_kwh"] = df.apply(generar_consumo, axis=1)


# 4. Etiquetado usando la lógica del Backend

def clasificar_perfil(fila):
    baseline = baseline_por_tipo[fila["tipo_inmueble"]]
    baseline += fila["cantidad_equipos"] * 8
    baseline *= factor_clima[fila["moneda_region"]]

    ratio = fila["consumo_kwh"] / max(baseline, 1)

    if fila["uso_horario_pico"] == 1:
        ratio *= 1.15
    if fila["horas_alto_consumo"] > 8:
        ratio *= 1.10

    # Penalización adicional leve si es fin de semana (dia_semana >= 5) con uso prolongado

    if fila["dia_semana"] >= 5 and fila["horas_alto_consumo"] > 6:
        ratio *= 1.05

    if ratio > 1.35:
        return "Ineficiente"
    elif ratio > 1.05:
        return "Moderado"
    else:
        return "Eficiente"

df["perfil_energetico"] = df.apply(clasificar_perfil, axis=1)

- Visualizacion de base de datos simulados

In [23]:
df.head()

,consumidor,tipo_inmueble,moneda_region,mes,dia_semana,uso_horario_pico,cantidad_equipos,horas_alto_consumo,consumo_kwh,perfil_energetico
0,Usuario_1,Oficina,BRL,6,1,1,17,1,379.63,Eficiente
1,Usuario_2,Comercio,CLP,4,2,1,27,6,518.57,Eficiente
2,Usuario_3,Casa,CLP,4,1,0,18,4,499.45,Moderado
3,Usuario_4,Comercio,PEN,2,4,1,19,6,1096.00,Ineficiente
4,Usuario_5,Oficina,ARS,3,3,1,12,7,548.82,Eficiente


- Guardado de DataFrame en formato CVS

In [19]:
df.to_csv("dataset_inmuebles.csv", index=False)